# 16.1 图基础与 NetworkX / Graph Basics & NetworkX

**中文**：前面所有数据都是"表格"（每行一个独立样本）或"序列"。但现实世界充满**关系**：社交网络里人与人是朋友、论文之间相互引用、网页之间相互链接、分子里原子相互成键。这些数据的核心不是"每个样本长什么样"，而是"**谁和谁相连**"。**图（graph）** 就是表达关系的数学结构，本部分(Part 16)从图论基础一路讲到图神经网络(GNN)。
**English**: All data so far has been "tabular" (each row an independent sample) or "sequential." But the real world is full of **relationships**: people are friends in social networks, papers cite each other, web pages link to each other, atoms bond in molecules. The essence of such data is not "what each sample looks like" but "**who connects to whom**." A **graph** is the mathematical structure for relationships; this part (Part 16) goes from graph-theory basics all the way to Graph Neural Networks (GNNs).

---

**中文**：一个图 $G=(V,E)$ 由两部分组成：
**English**: A graph $G=(V,E)$ has two parts:
- **节点 / nodes (vertices) $V$**：实体（人、论文、网页、原子）。
  **Nodes (vertices) $V$**: entities (people, papers, web pages, atoms).
- **边 / edges $E$**：实体之间的关系（朋友、引用、链接、化学键）。
  **Edges $E$**: relationships between entities (friendship, citation, link, bond).

**中文**：图有几个关键的区分维度：
**English**: Graphs differ along a few key axes:
- **有向 vs 无向 / directed vs undirected**：朋友关系是双向的(无向)，"A 关注 B"是单向的(有向)。
  Friendship is mutual (undirected); "A follows B" is one-way (directed).
- **带权 vs 无权 / weighted vs unweighted**：边上是否有数值(如通话时长、距离)。
  Whether edges carry a number (call duration, distance).

> 💡 **面试速查 / Interview cheat-sheet（★★ 基础但必问）**
> **中文**：图 $G=(V,E)$；**度(degree)** = 一个节点连了多少条边(有向图分入度/出度)。两种核心存储：**邻接矩阵(adjacency matrix)** $A\in\mathbb R^{n\times n}$，$A_{ij}=1$ 表示 $i,j$ 相连——查询 O(1) 但占 $O(n^2)$ 空间，**适合稠密图**；**邻接表(adjacency list)** 每个节点存一个邻居列表——占 $O(n+m)$ 空间($m$=边数)，**适合稀疏图**(现实图几乎都稀疏)。无向图的 $A$ 对称。
> **English**: Graph $G=(V,E)$; **degree** = how many edges a node has (in/out-degree for directed). Two core storages: **adjacency matrix** $A\in\mathbb R^{n\times n}$, $A_{ij}=1$ if $i,j$ connected — O(1) lookup but $O(n^2)$ space, **good for dense graphs**; **adjacency list** stores a neighbor list per node — $O(n+m)$ space ($m$=#edges), **good for sparse graphs** (real graphs are almost always sparse). For undirected graphs $A$ is symmetric.


In [ ]:

# ============================================================
# 数据集：扎卡里空手道俱乐部 / Zachary's Karate Club
# 中文：这是图论/社交网络分析最经典的"Hello World"数据集(Zachary 1977)。
#       34 个成员(节点)，边表示两人在俱乐部外也有社交往来。
#       传奇之处：俱乐部因管理员(node 0, Mr.Hi)与教练(node 33, Officer)闹翻而分裂成两派，
#       而**仅凭社交网络结构**就能预测每个人会站哪一队——后面社区发现会验证。
# English: The classic "Hello World" of graph/social-network analysis (Zachary 1977).
#       34 members (nodes); an edge means the two socialized outside the club.
#       Famous because the club split into two factions (admin node 0 vs coach node 33),
#       and the split is predictable from network structure alone (we verify in community detection).
# ============================================================
import networkx as nx, numpy as np, matplotlib.pyplot as plt
np.random.seed(0)

G = nx.karate_club_graph()                     # 内置数据集 / built-in dataset
print("节点数 / #nodes:", G.number_of_nodes())
print("边数   / #edges:", G.number_of_edges())
print("是否有向 / directed?", G.is_directed())
print("前 5 条边 / first 5 edges:", list(G.edges())[:5])
# 每个成员有一个真实派系标签('Mr. Hi' 或 'Officer')，是分裂后的归属 / ground-truth faction label
print("节点0的属性 / node 0 attrs:", G.nodes[0])


**中文**：图最基本的描述量是**度（degree）**——每个节点连了多少条边。度分布往往**极不均匀**：少数"中心人物"度很高，大多数节点度很低。这正是现实网络的普遍特征（社交、互联网、引用网络都呈现这种"长尾"/幂律倾向）。
**English**: The most basic descriptor is **degree** — how many edges each node has. The degree distribution is usually **highly uneven**: a few "hubs" have high degree, most nodes are low-degree. This is a universal trait of real networks (social, internet, citation networks all show this long-tail / power-law tendency).


In [ ]:

# ============================================================
# 度、邻居、邻接矩阵 / degree, neighbors, adjacency matrix
# ============================================================
deg = dict(G.degree())                                    # 每个节点的度 / degree per node
top = sorted(deg.items(), key=lambda kv: kv[1], reverse=True)[:5]
print("度最高的 5 个节点 / top-5 by degree:", top)        # node 0(Mr.Hi)、33(Officer) 应在前列
print("节点0的邻居 / neighbors of node 0:", sorted(G.neighbors(0)))

# 邻接矩阵 A：A[i,j]=1 表示 i,j 相连；无向图对称 / adjacency matrix, symmetric for undirected
A = nx.to_numpy_array(G, weight=None)                    # (34,34) 的 0/1 矩阵(忽略边权) / 0-1 matrix (ignore weights)
print("\n邻接矩阵形状 / A shape:", A.shape, "| 是否对称 symmetric:", np.allclose(A, A.T))
print("A 中 1 的个数 / #ones:", int(A.sum()), "= 2 × 边数 (无向边各算两次) / = 2×#edges")
print("节点i的度 = A第i行之和 / degree(i) = row sum:", int(A[0].sum()), "== deg[0]:", deg[0])
# 稀疏度：真实图绝大多数格子是 0 / sparsity: most cells are 0
print(f"邻接矩阵稠密度 / density: {A.mean():.3f}  (即 {1-A.mean():.1%} 是 0)")


**中文**：图最直观的理解方式是**画出来**。NetworkX 用"力导向布局（spring layout）"把图摆开——把边看成弹簧、节点互相排斥，迭代到平衡，于是**结构紧密相连的节点会聚在一起**。我们按度给节点上色、调大小，并用真实派系标签区分形状。
**English**: The most intuitive way to grasp a graph is to **draw it**. NetworkX uses a "spring layout" — treat edges as springs and nodes as mutually repelling, iterate to equilibrium, so **densely connected nodes cluster together**. We color/size nodes by degree and mark the two ground-truth factions.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
pos = nx.spring_layout(G, seed=42)                         # 力导向布局(固定种子可复现) / spring layout

# ① 按度上色与缩放 / color & size by degree
sizes = [deg[i]*60 for i in G.nodes()]                     # 度越大点越大 / bigger = higher degree
nodes = nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=[deg[i] for i in G.nodes()],
                               cmap="viridis", ax=ax[0])
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax[0])
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax[0])
ax[0].set_title("空手道俱乐部：节点大小/颜色=度 / size&color = degree"); ax[0].axis("off")
plt.colorbar(nodes, ax=ax[0], fraction=0.04, label="degree")

# ② 按真实派系上色 / color by ground-truth faction
clr = ["#4C72B0" if G.nodes[i]["club"]=="Mr. Hi" else "#C44E52" for i in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_size=300, node_color=clr, ax=ax[1])
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax[1])
nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", ax=ax[1])
ax[1].set_title("两大派系(分裂后真实归属) / two real factions"); ax[1].axis("off")
plt.tight_layout(); plt.savefig("/tmp/g01_viz.png", dpi=80); plt.show()
print("观察：度高的 0 和 33 正是两派的核心(管理员与教练)/ hubs 0 & 33 are the two faction cores")


**中文**：再认识几个**全局描述量**，它们刻画"整张图长什么样"：
**English**: A few more **global descriptors** characterizing "what the whole graph looks like":

**中文**：
- **密度 density** = 实际边数 / 最多可能边数，衡量图有多"稠密"。
- **聚类系数 clustering coefficient**：你的两个朋友也互为朋友的概率("物以类聚")——社交网络通常很高。
- **平均最短路径 / 直径 diameter**：任意两点平均要走几步 / 最远要走几步——现实网络往往很小("六度分隔")。
- **连通分量 connected components**：图是否连成一片。

**English**:
- **Density** = actual edges / max possible edges — how "dense" the graph is.
- **Clustering coefficient**: the probability that two of your friends are also friends ("triadic closure") — usually high in social networks.
- **Average shortest path / diameter**: average / maximum hops between any two nodes — often surprisingly small in real networks ("six degrees of separation").
- **Connected components**: whether the graph is one connected piece.


In [ ]:

# ============================================================
# 全局结构指标 / global structural metrics
# ============================================================
print(f"密度 density: {nx.density(G):.3f}  (= 2m / [n(n-1)])")
print(f"平均聚类系数 avg clustering: {nx.average_clustering(G):.3f}  (朋友的朋友也是朋友的概率)")
print(f"连通分量数 #connected components: {nx.number_connected_components(G)}")
if nx.is_connected(G):                                     # 连通才能算路径 / paths need connectivity
    print(f"平均最短路径 avg shortest path: {nx.average_shortest_path_length(G):.3f} 步/hops")
    print(f"直径 diameter (最远两点) / farthest pair: {nx.diameter(G)} 步/hops")
# 与同规模随机图对比聚类系数 / compare clustering to a random graph of same size
n, m = G.number_of_nodes(), G.number_of_edges()
rand = nx.gnm_random_graph(n, m, seed=1)                   # 随机连相同数量的边 / random graph
print(f"\n同规模随机图聚类系数 / random-graph clustering: {nx.average_clustering(rand):.3f}")
print("→ 真实社交图聚类远高于随机图(小世界特征) / real graph is far more clustered (small-world)")


**中文**：诚实地看几个观察点：
**English**: A few honest observations:

**中文**：
1. **度极不均匀**：node 0(管理员)和 node 33(教练)的度远高于其他人——他们是网络的"枢纽"，也是两派的核心。这预示了后面用结构就能预测派系。
2. **高聚类、短路径 = 小世界**：真实空手道图的聚类系数(~0.57)远高于同规模随机图(~0.15)，但平均路径依然很短(~2.4 步)。这就是著名的"**小世界(small-world)**"现象——朋友圈高度抱团，但全图却任意两人几步可达。
3. **稀疏**：邻接矩阵 90%+ 是 0。这就是为什么大图要用**邻接表/稀疏矩阵**存储，也是后面 GNN 用"邻居聚合"而非稠密矩阵运算的根本原因。

**English**:
1. **Highly uneven degree**: node 0 (admin) and node 33 (coach) have far higher degree — the network's "hubs" and the cores of the two factions, foreshadowing that structure alone predicts the split.
2. **High clustering + short paths = small world**: the real karate graph's clustering (~0.57) far exceeds a same-size random graph (~0.15), yet average path length stays short (~2.4 hops). This is the famous "**small-world**" phenomenon — tight local clusters, yet everyone reachable in few steps.
3. **Sparse**: the adjacency matrix is 90%+ zeros. This is why large graphs use **adjacency lists / sparse matrices**, and why GNNs later aggregate over neighbors rather than doing dense matrix ops.

> 💼 **实战视角 / Practical angle**
> **中文**：图无处不在——推荐(用户-物品二部图)、反欺诈(交易网络找团伙)、知识图谱、分子性质预测、社交关系挖掘、地图导航。**第一步永远是想清楚"节点是什么、边是什么、有向吗、带权吗"**——建模图问题，建图比选算法更关键。工具上 **NetworkX** 适合中小图分析与原型(本部分主力)；超大图用 **igraph/graph-tool**(C 底层)或图数据库 **Neo4j**；深度学习用 **PyTorch Geometric / DGL**(本仓未装，我们从零实现以讲透原理)。
> **English**: Graphs are everywhere — recommendation (user-item bipartite graph), fraud detection (find rings in transaction networks), knowledge graphs, molecular property prediction, social mining, map navigation. **The first step is always defining "what are nodes, what are edges, directed?, weighted?"** — modeling the graph matters more than picking the algorithm. Tooling: **NetworkX** for small/medium analysis and prototyping (our workhorse); **igraph/graph-tool** (C backends) or **Neo4j** for huge graphs; **PyTorch Geometric / DGL** for deep learning (not installed here, so we implement from scratch to teach the mechanics).

---
### 小结 / Summary
- **中文**：图 $G=(V,E)$ 表达关系；核心量是度、邻接矩阵/邻接表、密度、聚类系数、最短路径。
- **English**: A graph $G=(V,E)$ encodes relationships; core quantities are degree, adjacency matrix/list, density, clustering, shortest paths.
- **中文**：现实图普遍**稀疏 + 小世界**(高聚类、短路径、度长尾)——决定了存储与算法的选择。
- **English**: Real graphs are **sparse + small-world** (high clustering, short paths, long-tail degree) — dictating storage and algorithm choices.
- **中文**：空手道俱乐部的结构里已"藏着"两派之分，后续章节将逐步把它挖出来。
- **English**: The karate club's structure already "hides" the two factions, which later chapters progressively uncover.
